# IPC2BNS-Verify — Phase 2: Ingestion & Retrieval Layer

This notebook verifies Phase 2 deliverables:
1. **Section-Level Chunker** (`chunker.py`): Structured chunks with temporal validity metadata.
2. **Cleaned Corpora**: `ipc_sections.jsonl` (145 provisions) and `bns_sections.jsonl` (130 provisions).
3. **Statutory Vector Index** (`embedder.py`): BM25/hybrid similarity engine.
4. **Retrieval Search Engine** (`search.py`): Top-k retrieval with temporal validity filtering.
5. **Benchmark Dataset Evaluation** (`retrieval_eval.py`): Computes Recall@k, Precision@k, and MRR.
6. **Automated Pytest Suite**: Full test run.

---
## 1. Mount Google Drive & Configure Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project Root:', PROJECT_ROOT)
print('Environment initialized.')

Mounted at /content/drive
Project Root: /content/drive/MyDrive/NLP_rspaper
Environment initialized.


---
## 2. Dependencies

In [2]:
!pip install -q pytest
print('Pytest ready.')

Pytest ready.


---
## 3. Inspect Cleaned Statutory Corpora & Chunks

In [3]:
from src.ingestion.chunker import load_all_chunks

cleaned_dir = os.path.join(PROJECT_ROOT, 'data/01_cleaned')
chunks_dict = load_all_chunks(cleaned_dir)

print(f'IPC Chunks Loaded: {len(chunks_dict["IPC"])}')
print(f'BNS Chunks Loaded: {len(chunks_dict["BNS"])}')
print(f'Total Corpus Size: {len(chunks_dict["ALL"])} sections\n')

# Sample chunk
sample = chunks_dict['BNS'][0]
print('--- Sample Statutory Chunk ---')
print(f'ID      : {sample.chunk_id}')
print(f'Act     : {sample.act_full_name}')
print(f'Section : §{sample.section_number} - {sample.section_title}')
print(f'Dates   : {sample.effective_start} to {sample.effective_end}')
print(f'Text    : {sample.section_text[:120]}...')

IPC Chunks Loaded: 145
BNS Chunks Loaded: 130
Total Corpus Size: 275 sections

--- Sample Statutory Chunk ---
ID      : BNS_SEC_1
Act     : Bharatiya Nyaya Sanhita, 2023
Section : §1 - Short title commencement and application
Dates   : 2024-07-01 to 9999-12-31
Text    : Statutory provision for BNS Section 1: Short title commencement and application. Merged into BNS Section 1...


---
## 4. Build / Load Statutory Vector Index

In [4]:
from src.retrieval.embedder import build_and_save_index, LocalStatutoryVectorIndex

index_dir = os.path.join(PROJECT_ROOT, 'data/05_embeddings_index/stage2_index')
index = build_and_save_index(cleaned_dir, index_dir)
print('Vector index successfully built and persisted.')

Vector index successfully built and persisted.


---
## 5. Interactive Statutory Retrieval Queries

In [5]:
from src.retrieval.search import retrieve_statutes

queries = [
    ('What is the punishment for murder under BNS?', 'BNS'),
    ('Where is snatching or petty theft penalized?', 'BNS'),
    ('Penalty for rash driving causing death (hit and run)?', 'BNS'),
    ('What section defines cheating in IPC?', 'IPC'),
    ('Provisions for terrorist acts under new law?', 'BNS'),
]

for q, act in queries:
    print('='*75)
    print(f'Query: "{q}" [Filter: {act}]')
    print('='*75)
    hits = retrieve_statutes(q, top_k=2, act_filter=act)
    for rank, h in enumerate(hits, start=1):
        print(f'  [{rank}] {h["act"]} §{h["section_number"]}: {h["section_title"]} (score: {h["similarity_score"]:.2f})')
        print(f'      Text: {h["section_text"][:100]}...')
    print()

Query: "What is the punishment for murder under BNS?" [Filter: BNS]
  [1] BNS §103: Punishment for murder (score: 42.54)
      Text: (1) Whoever commits murder shall be punished with death or imprisonment for life, and shall also be ...
  [2] BNS §101: Murder (score: 34.04)
      Text: Statutory provision for BNS Section 101: Murder....

Query: "Where is snatching or petty theft penalized?" [Filter: BNS]
  [1] BNS §303(2): Snatching (score: 40.30)
      Text: Statutory provision for BNS Section 303(2): Snatching. New sub-section specifically criminalizing sn...
  [2] BNS §112: Petty organised crime (score: 18.08)
      Text: (1) Whoever, being a member of a group or gang, commits theft, snatching, cheating, unauthorized sel...

Query: "Penalty for rash driving causing death (hit and run)?" [Filter: BNS]
  [1] BNS §281: Rash driving or riding on a public way (score: 58.60)
      Text: Statutory provision for BNS Section 281: Rash driving or riding on a public way....
  [2] BNS §38: When

---
## 6. Temporal Validity Filtering Demonstration

Demonstrates the **TaxFlow-inspired temporal validity filtering**: queries set before July 1, 2024 retrieve IPC, while queries set after retrieve BNS.

In [6]:
from src.retrieval.search import get_retriever
retriever = get_retriever()

print('--- Conduct Date: 2021-05-15 (Pre-Transition -> IPC Applies) ---')
hits_2021 = retriever.retrieve('murder', top_k=2, target_date='2021-05-15')
for h in hits_2021:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]}')

print('\n--- Conduct Date: 2025-01-10 (Post-Transition -> BNS Applies) ---')
hits_2025 = retriever.retrieve('murder', top_k=2, target_date='2025-01-10')
for h in hits_2025:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]}')

--- Conduct Date: 2021-05-15 (Pre-Transition -> IPC Applies) ---
  IPC §300: Murder
  IPC §304: Punishment for culpable homicide not amounting to murder

--- Conduct Date: 2025-01-10 (Post-Transition -> BNS Applies) ---
  BNS §101: Murder
  BNS §105: Punishment for culpable homicide not amounting to murder


---
## 7. Evaluate Retrieval Accuracy (Precision, Recall, MRR)

In [7]:
from src.eval.retrieval_eval import evaluate_retrieval

benchmark_dev = os.path.join(PROJECT_ROOT, 'data/03_benchmark/benchmark_dev.csv')
metrics_out = os.path.join(PROJECT_ROOT, 'results/stage2/retrieval_metrics.json')

metrics = evaluate_retrieval(benchmark_dev, metrics_out, top_k=5)

print('\n' + '='*50)
print('STAGE 2 RETRIEVAL METRICS SUMMARY')
print('='*50)
print(json.dumps(metrics['metrics'], indent=2))


STAGE 2 RETRIEVAL METRICS SUMMARY
{
  "recall_at_1": 0.5294,
  "recall_at_3": 0.7647,
  "recall_at_5": 0.7647,
  "precision_at_1": 0.5294,
  "precision_at_3": 0.2549,
  "precision_at_5": 0.1529,
  "mean_reciprocal_rank": 0.6471,
  "avg_latency_ms": 2.09
}


---
## 8. Run Full Automated Test Suite

In [8]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.1
collected 50 items                                                             

drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_table_loads_successfully PASSED [  2%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_schema_columns PASSED [  4%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[302-103] PASSED [  6%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[299-100] PASSED [  8%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[304A-106] PASSED [ 10%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[30

---
## 9. Check WBS Completion

In [9]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report

# Project Progress Report
**Overall: 14/32 tasks complete (44%)**

_Generated: 2026-09-03T07:02:12_

## 0. Setup — 3/4 (75%)
- [x] Repo scaffolding + config system  `(code/src, code/configs)`
- [x] India Code raw text downloaded  `(data/00_raw/india_code)`
- [ ] Concordance source PDF(s) collected  `(data/00_raw/concordance_source_pdfs)`
- [x] Data Management Plan written  `(docs/IPC2BNS-Verify_Data_Management_Plan.md)`

## 1. Mapping Module — 5/5 (100%)
- [x] Ground-truth concordance table finalized  `(data/02_ground_truth/concordance_v1.csv)`
- [x] Concordance validation report reviewed  `(data/02_ground_truth/validation_report.csv)`
- [x] Deterministic lookup function implemented  `(code/src/mapping/lookup.py)`
- [x] Query normalizer implemented  `(code/src/mapping/normalizer.py)`
- [x] Mapping module unit tests  `(code/tests/test_concordance.py)`

## 2. Ingestion & Retrieval — 6/6 (100%)
- [x] Section-level chunker implemented  `(code/src/ingestion/chunker.py)`
- [x] Cleaned sectio